## Task 0: Classical Skeleton

A 3x3 grid represented as a list of 9 cells, each either empty, 'X', or 'O'

A game loop that alternates Player 1 (X) and Player 2 (O)

In [1]:
# quantum_ttt.py
#The board starts as ['1', '2',...,'9'] and as players make moves, the numbers are replaced with 'X' or 'O'.
# A number on the board represents an available move, while 'X' or 'O' represents a move made by a player.
board = [str(i) for i in range(1, 10)] #['1', '2',.....'9']

def display_board():
    print()
    print(' ' + board[0] + ' | ' + board[1] + ' | ' + board[2])
    print('---+---+---')
    print(' ' + board[3] + ' | ' + board[4] + ' | ' + board[5])
    print('---+---+---')
    print(' ' + board[6] + ' | ' + board[7] + ' | ' + board[8])
    print() #Print the 3*3 grid with the current state of the board.


def game_loop():
    players = ["X", "O"] #Two players, 'X' and 'O'.
    
    for turn in range(9): #Maximum of 9 turns in a game of tic-tac-toe.
        display_board() #Display the current state of the board at the start of each turn.

        player = players[turn %2] #Alternate between player 'X' and 'O' based on the turn number.

        while True:
            choice = input(f"Player{player}, pick a square(1-9)")

            # if the choice is still in the board, the cell is empty and the input is valid.
            # if it's been replaced by 'X' od 'O', or doesn't exist, it fails

            if choice in board:
                board[int(choice) - 1] = player #replace the number with X or O
                break #valid move made, exit the while loop
            print("Invalid move. Try again.") #Prompt the player to try again if the move is invalid.


        # Show the final board state after all turns
        display_board()

    game_loop()



## Task 1: The Quantum Backend

This is a quantum circuit with 9 qubits and 9 classical bits for eventual measurement.

Also write a helper that links the python board to the circuit


In [2]:
from qiskit import QuantumCircuit

""" I need 9 qubits for each square on the board, think of each qubit as the quantum state of that square"""

qc = QuantumCircuit(9, 9)
qc.draw('mpl')

def init_quantum_backend():
    # This function will initialize the quantum backend, which is necessary for running quantum circuits.
    print("Quantum backend initialized.")

def square_to_qubit(square):
    # This function will map a square number (1-9) to a corresponding qubit index (0-8).
    return int(square) - 1
# Call this once at the start of the game
init_quantum_backend()

# Test the helper function
for square in range(1, 10):
    qubit_index = square_to_qubit(square)
    print(f"Square {square} maps to qubit index {qubit_index}.")




Quantum backend initialized.
Square 1 maps to qubit index 0.
Square 2 maps to qubit index 1.
Square 3 maps to qubit index 2.
Square 4 maps to qubit index 3.
Square 5 maps to qubit index 4.
Square 6 maps to qubit index 5.
Square 7 maps to qubit index 6.
Square 8 maps to qubit index 7.
Square 9 maps to qubit index 8.


## Task 2: Implementing "Spooky" Moves
 In Quantum Tic-Tac-Toe a player picks two squares — their mark exists in both at the same time (superposition). Nobody knows which one it will actually land on until the state collapses later.

 To represent this in Qiskit:

A Hadamard gate on a qubit puts it into superposition — it becomes both 0 and 1 at the same time, representing the move being "uncertain"

A CNOT gate entangles two qubits together — meaning the two squares the player picked are now linked. Whatever happens to one affects the other

In [3]:
superposition_moves = []

#counts moves made, used to label marks e.g X1, X2, O1, O2, etc.
move_counter = 0

def get_two_squares(board):
    # Ask the player to pick two squares for their move
    squares = []
    for i in range("first", "second"):
        while True:
            try:
                sq = int(input(f" Pick your {i} square (1-9): "))
                if 1 <= sq <= 9 and str(sq) in board:
                    squares.append(str(sq))
                    break
                else:
                    print("Invalid choice. Try again.")
            except ValueError:
                print("Please enter a valid number.")
    return squares[0], squares[1]

def apply_quantum_gates(q1, q2):
    # This function will apply quantum gates to the qubits corresponding to the chosen squares.
    # For simplicity, we will just create a superposition of the two chosen squares.
    qc.h(q1)  # Apply Hadamard gate to the first qubit
    qc.cx(q1, q2)  # Apply CNOT gate to entangle the two qubits


def make_spooky_move(board, player):
    """A spooky move lets the player place their mark in TWO squares at once.
    The mark exists in both squares simultaneously until collapse in Task 3 """
    global move_counter
    move_counter += 1

    # Get two squares from the player
    sq1, sq2 = get_two_squares(board)

    # Convert squares (1-9) to qubit indices (0-8) using the helper
    q1, q2 = square_to_qubit(sq1), square_to_qubit(sq2)

    #Apply Hadamard + CNOT to entangle the two qubits
    apply_quantum_gates(q1, q2)

    #label for this move e.g X1, X2, O1, O2, etc.
    label = f"{player}{move_counter}"

    # update the board with the label for both squares
    board[sq1-1] = label
    board[sq2-1] = label

    #Store the move for the superposition collapse in Task 3
    superposition_moves.append({'label': label, 'squares': (sq1, sq2)})

    print(f" {label} placed in squares {sq1} and {sq2}.\n")

## Task 3: State Collapse



Collapse is the moment reality is forced to choose- each superposition mark lands on exactly one of its two squares



In [ ]:
from qiskit_aer import AerSimulator

def detect_loop(superposition_moves):
    # This function will check for loops in the superposition moves.
    # A loop occurs when a move creates a cycle of entanglement that cannot be resolved.
    
    """A loop means a cycle exists where squares are shared between moves.
    Example of a loop:
        X1 on squares (1, 5)
        O1 on squares (5, 9)
        X2 on squares (9, 1)
        Square 1 -> 5 -> 9 -> 1 forms a cycle — collapse must happen!
    
    We detect this by building a graph where:
        - Each square is a node
        - Each superposition move is an edge between two squares
    A loop exists when any node is visited twice."""

    graph ={}
    sq1, sq2 = move['squares']
    graph.setdefault(sq1, []).append(sq2)
    graph.setdefault(sq2, []).append(sq1)

    # Check the graph to detect a cycle using simple visited tracking
    visited = set()
    
     